# Natural Latents: Complete Comparison Notebook

**Every combination tested:** 2 embedding schemes (generic char n-gram vs.
domain-specific tokenizers) x 2 resampling states (with/without) x 5 loss
measures (KL divergence, cross-entropy, rank-based info gain, signed
log-likelihood ratio, and PMI as a deliberate negative control) x a combined
"mix" of all four real directional measures together.

## Thought process, up front

Every step shows you the actual data at that point -- not just a final
number. The design follows exactly what worked best earlier in this
project: losses are computed **independently per representation** (Lean,
LaTeX, Python lambda, Cayley table, natural language) and fed to a trained
classifier as separate features, rather than being pooled into one blended
score before comparison -- this "late fusion" approach was found to
roughly double accuracy versus blending representations together first.


## 0. Setup

In [ ]:

import sys, os, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              confusion_matrix, precision_recall_fscore_support)

sys.path.insert(0, '../src')
sys.path.insert(0, '../data')
from embeddings import make_generic_embedder, make_domain_specific_embedder
from resampling import NaturalLatentResampler
from losses import LOSS_FUNCTIONS

DATA_DIR = '../data'
np.random.seed(42)
random.seed(42)
print("Setup complete.")


## 1. Load real data

Real equations and ground truth from the Equational Theories Project
(Terence Tao), via `corpaci/semantic-diffchecking`. Using a sample of 500
equations for a notebook that runs end-to-end in a reasonable time --
the full-scale version of this exact pipeline (4,612 equations, 21M pairs)
is documented separately; this notebook reproduces the same methodology
at a size you can run and inspect directly.


In [ ]:

with open(os.path.join(DATA_DIR, 'equations_representations.json')) as f:
    all_equations = json.load(f)

# keep only equations with ALL 5 representations present
all_equations = [e for e in all_equations if e['natural_language'] is not None]
print(f"Equations with all 5 representations: {len(all_equations)}")

# FIX: random sample, not "first N" -- avoids any ordering bias in the source catalogue
SAMPLE_SIZE = 1200
random.seed(42)
sample_equations = random.sample(all_equations, SAMPLE_SIZE)
sample_nodes_list = [e['node'] for e in sample_equations]   # ordered list, used for direct pair generation below
sample_nodes = set(sample_nodes_list)
print(f"Using a RANDOM sample of {len(sample_equations)} equations for this notebook")

print("\n--- VIEW ON THE DATA: one real equation, all 5 representations ---")
example = sample_equations[10]
for k, v in example.items():
    print(f"{k:20s}: {str(v)[:100]}")


In [ ]:

# FIX: generate pairs DIRECTLY from the sampled equations via the exact relationship
# matrix, instead of hoping a generic pre-sampled file happens to intersect with our
# sample. This was the main fix -- confirmed to give ~100x-200x more usable pairs at
# essentially zero extra compute cost.
from decode_matrix import load_matrix, relation_for

matrix, meta = load_matrix()
rows = []
for i, a in enumerate(sample_nodes_list):
    for b in sample_nodes_list[i+1:]:
        rel = relation_for(matrix, a, b)
        if rel != 'unresolved':
            rows.append({'node_a': a, 'node_b': b, 'relation': rel})

all_available = pd.DataFrame(rows)
print(f"Pairs available directly from this sample: {len(all_available):,}")

PAIR_SAMPLE_SIZE = 15000
pairs = all_available.sample(n=min(PAIR_SAMPLE_SIZE, len(all_available)), random_state=42).reset_index(drop=True)
print(f"Using {len(pairs):,} pairs for this notebook run")
print(pairs['relation'].value_counts())
print("\n--- VIEW ON THE DATA: first 5 labeled pairs ---")
pairs.head()


## 2. Build BOTH embedding schemes

Generic (char n-grams, the scheme behind the ~46-47% result) and
domain-specific (a real tokenizer per representation type: Lean
keywords/symbols, actual Python syntax tokens via `tokenize`, LaTeX
commands, and word-level for natural language).


In [ ]:

representations = {}
for view in ['lean', 'latex', 'py_lambda', 'py_cayley_table', 'natural_language']:
    representations[view] = {e['node']: e[view] for e in sample_equations}

print("--- VIEW: representations dict structure ---")
print("views:", list(representations.keys()))
print("equations per view:", len(representations['lean']))


In [ ]:

embed_generic = make_generic_embedder(char_ngrams=(2,4), max_features=500)
embed_domain = make_domain_specific_embedder(max_features=500)

resampler_generic = NaturalLatentResampler(embed_generic, ridge_alpha=1.0, n_components=8)
resampler_generic.fit_embeddings(representations)

resampler_domain = NaturalLatentResampler(embed_domain, ridge_alpha=1.0, n_components=8)
resampler_domain.fit_embeddings(representations)

print("--- VIEW: embedding dimensionality per view, both schemes ---")
for view in representations:
    print(f"{view:20s} generic={resampler_generic.view_matrices[view].shape}  "
          f"domain_specific={resampler_domain.view_matrices[view].shape}")


## 3. Correlation heatmap -- the finding that motivated "late fusion"

Earlier in this project we found the four formal representations
(Lean/LaTeX/Python-lambda/Cayley-table) correlate moderately with each
other, while natural language correlates close to zero with all of them.
Here's that finding, shown directly rather than just described.


In [ ]:

def view_correlation_heatmap(view_matrices, title):
    views = list(view_matrices.keys())
    # reduce each view to its mean-pooled scalar profile for a simple,
    # interpretable correlation: correlate each view's OWN pairwise distance
    # structure against every other view's pairwise distance structure
    from scipy.spatial.distance import pdist
    profiles = {v: pdist(view_matrices[v]) for v in views}
    corr = np.zeros((len(views), len(views)))
    for i, vi in enumerate(views):
        for j, vj in enumerate(views):
            corr[i,j] = np.corrcoef(profiles[vi], profiles[vj])[0,1]

    plt.figure(figsize=(6,5))
    sns.heatmap(corr, xticklabels=views, yticklabels=views, annot=True, fmt='.2f',
                cmap='RdBu_r', vmin=-1, vmax=1, square=True)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'../outputs/correlation_heatmap_{title.replace(" ","_")}.png', dpi=100)
    plt.show()
    return views, corr

views, corr_generic = view_correlation_heatmap(resampler_generic.view_matrices, "Generic embedding")
print("\nNatural language row (generic embedding):", dict(zip(views, np.round(corr_generic[views.index('natural_language')],3))))


In [ ]:

views_d, corr_domain = view_correlation_heatmap(resampler_domain.view_matrices, "Domain-specific embedding")
print("\nNatural language row (domain-specific embedding):", dict(zip(views_d, np.round(corr_domain[views_d.index('natural_language')],3))))


## 4. Build per-view latents, WITH and WITHOUT resampling, both embedding schemes

Four latent sets total: {generic, domain-specific} x {raw, resampled}.
Each keeps all 5 views SEPARATE (not pooled) -- this is what feeds the
per-view loss computation in the next section.


In [ ]:

print("Building resampled (denoised) latents -- this runs Ridge regression per view...")
t0 = time.time()
latents_generic_resampled = resampler_generic.per_view_resampled_latents(n_components=8)
latents_generic_raw = resampler_generic.raw_per_view_latents(n_components=8)
print(f"  generic scheme done in {time.time()-t0:.1f}s")

t0 = time.time()
latents_domain_resampled = resampler_domain.per_view_resampled_latents(n_components=8)
latents_domain_raw = resampler_domain.raw_per_view_latents(n_components=8)
print(f"  domain-specific scheme done in {time.time()-t0:.1f}s")

print("\n--- VIEW: one equation's purified latent, Lean view, generic embedding, resampled ---")
example_node = sample_equations[10]['node']
print(np.round(latents_generic_resampled['lean'][example_node], 3))


## 5. Compute all directional losses, independently per view

For every (embedding scheme, resampling state) combination, compute all 5
loss/divergence matrices, independently for each of the 5 views -- 5
views x 5 loss types = 25 separate NxN matrices per combination, 100 total.


In [ ]:

def compute_all_losses(latents_dict, node_order):
    '''latents_dict: view_name -> {node: vector}. Returns view_name -> loss_name -> NxN matrix.'''
    result = {}
    for view, node_latents in latents_dict.items():
        mat = np.stack([node_latents[n] for n in node_order])
        result[view] = {}
        for loss_name, loss_fn in LOSS_FUNCTIONS.items():
            result[view][loss_name] = loss_fn(mat)
    return result

node_order = sorted(set(pairs.node_a) | set(pairs.node_b))
node_idx = {n: i for i, n in enumerate(node_order)}
print(f"Computing losses for {len(node_order)} equations across 4 (embedding x resampling) combos...")

t0 = time.time()
losses_generic_raw = compute_all_losses(latents_generic_raw, node_order)
print(f"  generic + raw done ({time.time()-t0:.1f}s)")
t0 = time.time()
losses_generic_resampled = compute_all_losses(latents_generic_resampled, node_order)
print(f"  generic + resampled done ({time.time()-t0:.1f}s)")
t0 = time.time()
losses_domain_raw = compute_all_losses(latents_domain_raw, node_order)
print(f"  domain-specific + raw done ({time.time()-t0:.1f}s)")
t0 = time.time()
losses_domain_resampled = compute_all_losses(latents_domain_resampled, node_order)
print(f"  domain-specific + resampled done ({time.time()-t0:.1f}s)")

print("\n--- VIEW: KL divergence, Lean view, generic+resampled, for one real pair ---")
i, j = node_idx[sample_equations[10]['node']], node_idx[sample_equations[20]['node']]
print(f"KL(eq_a | eq_b) = {losses_generic_resampled['lean']['KL'][i,j]:.3f}")
print(f"KL(eq_b | eq_a) = {losses_generic_resampled['lean']['KL'][j,i]:.3f}")


## 6. Build feature sets and labels for every combination


In [ ]:

label_map = {'equivalent': 0, 'stronger': 1, 'weaker': 2, 'incomparable': 3}
LABELS = [0,1,2,3]
NAMES = ['equivalent','stronger','weaker','incomparable']

ii = np.array([node_idx[a] for a in pairs.node_a])
jj = np.array([node_idx[b] for b in pairs.node_b])
y = np.array([label_map[r] for r in pairs.relation])

VIEW_NAMES = ['lean', 'latex', 'py_lambda', 'py_cayley_table', 'natural_language']
LOSS_NAMES = ['KL', 'CrossEntropy', 'RankInfoGain', 'SignedLLR', 'PMI_control']

def build_features(losses_dict, loss_name):
    '''Single loss type, all 5 views, both directions -- 10 features.'''
    cols = []
    for v in VIEW_NAMES:
        mat = losses_dict[v][loss_name]
        cols.append(mat[ii, jj])
        cols.append(mat[jj, ii])
    return np.stack(cols, axis=1)

def build_mix_features(losses_dict):
    '''All 4 real directional measures combined -- 5 views x 4 losses x 2 directions = 40 features.'''
    cols = []
    for loss_name in ['KL', 'CrossEntropy', 'RankInfoGain', 'SignedLLR']:
        for v in VIEW_NAMES:
            mat = losses_dict[v][loss_name]
            cols.append(mat[ii, jj])
            cols.append(mat[jj, ii])
    return np.stack(cols, axis=1)

print(f"Total pairs: {len(y)}")
print(f"Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")


## 7. Train + evaluate every combination

Node-based held-out split (not pair-based) -- equations in the test set
were never seen, in any pairing, during training. This is the same
discipline applied throughout this project specifically to avoid the kind
of leakage documented in the accompanying critique of the alternative
property-based approach.


In [ ]:

def node_holdout_split(node_order, ii, jj, test_frac=0.3, seed=42):
    rng = np.random.RandomState(seed)
    nodes = np.array(node_order)
    perm = rng.permutation(len(nodes))
    n_test = int(test_frac * len(nodes))
    test_nodes = set(nodes[perm[:n_test]].tolist())
    train_mask = np.array([node_order[i] not in test_nodes and node_order[j] not in test_nodes for i,j in zip(ii,jj)])
    test_mask = np.array([node_order[i] in test_nodes and node_order[j] in test_nodes for i,j in zip(ii,jj)])
    return train_mask, test_mask

# FIX: report accuracy across MULTIPLE independent splits, not one draw -- a single
# split with a small test set can be off by a wide margin (verified: with a 47-pair
# test set, the 95% confidence interval was +-14 points). 5 different seeds gives a
# real mean +- standard deviation instead of a number that could just be luck.
SPLIT_SEEDS = [42, 7, 123, 99, 256]

train_mask, test_mask = node_holdout_split(node_order, ii, jj, seed=SPLIT_SEEDS[0])
print(f"Example split (seed={SPLIT_SEEDS[0]}): Train pairs: {train_mask.sum():,}   Test pairs: {test_mask.sum():,}")
print(f"Will evaluate every combination across {len(SPLIT_SEEDS)} independent splits and report mean +- std.")


In [ ]:

def run_and_evaluate_one_split(X, y, train_mask, test_mask):
    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    clf = RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=10,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    acc = accuracy_score(y_test, pred)
    bal_acc = balanced_accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=LABELS)
    p, r, f, s = precision_recall_fscore_support(y_test, pred, labels=LABELS, zero_division=0)
    return acc, bal_acc, cm, p, r, f, s

def run_and_evaluate(X, y, node_order, ii, jj, label, seeds=SPLIT_SEEDS):
    '''Runs across multiple independent node-holdout splits. Returns mean +- std
    accuracy, a SUMMED confusion matrix (pooling all seeds\' test sets for a larger,
    more stable view), and mean precision/recall/f1.'''
    accs, bal_accs = [], []
    cm_total = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    p_list, r_list, f_list, s_total = [], [], [], np.zeros(len(LABELS))

    for seed in seeds:
        train_mask, test_mask = node_holdout_split(node_order, ii, jj, seed=seed)
        acc, bal_acc, cm, p, r, f, s = run_and_evaluate_one_split(X, y, train_mask, test_mask)
        accs.append(acc); bal_accs.append(bal_acc)
        cm_total += cm
        p_list.append(p); r_list.append(r); f_list.append(f); s_total += s

    return {
        'label': label,
        'accuracy': float(np.mean(accs)), 'accuracy_std': float(np.std(accs)),
        'balanced_accuracy': float(np.mean(bal_accs)), 'balanced_accuracy_std': float(np.std(bal_accs)),
        'confusion_matrix': cm_total,   # summed across all seeds -- a larger, more stable view
        'precision': np.mean(p_list, axis=0), 'recall': np.mean(r_list, axis=0), 'f1': np.mean(f_list, axis=0),
        'support': s_total, 'n_splits': len(seeds),
    }

results = []
combos = [
    ('Generic embedding, RAW (no resampling)', losses_generic_raw),
    ('Generic embedding, RESAMPLED', losses_generic_resampled),
    ('Domain-specific embedding, RAW (no resampling)', losses_domain_raw),
    ('Domain-specific embedding, RESAMPLED', losses_domain_resampled),
]

print(f"Evaluating all {len(combos)} x {len(LOSS_NAMES)+1} = {len(combos)*(len(LOSS_NAMES)+1)} combinations,")
print(f"each across {len(SPLIT_SEEDS)} independent splits ({len(combos)*(len(LOSS_NAMES)+1)*len(SPLIT_SEEDS)} total model fits)...\n")

for combo_label, losses_dict in combos:
    for loss_name in LOSS_NAMES:
        X = build_features(losses_dict, loss_name)
        full_label = f"{combo_label} | loss={loss_name}"
        results.append(run_and_evaluate(X, y, node_order, ii, jj, full_label))
        r = results[-1]
        print(f"done: {full_label}  ->  acc={r['accuracy']:.3f}+-{r['accuracy_std']:.3f}  balanced_acc={r['balanced_accuracy']:.3f}+-{r['balanced_accuracy_std']:.3f}")
    # the Mix combo
    X_mix = build_mix_features(losses_dict)
    full_label = f"{combo_label} | loss=Mix(KL+CE+Rank+LLR)"
    results.append(run_and_evaluate(X_mix, y, node_order, ii, jj, full_label))
    r = results[-1]
    print(f"done: {full_label}  ->  acc={r['accuracy']:.3f}+-{r['accuracy_std']:.3f}  balanced_acc={r['balanced_accuracy']:.3f}+-{r['balanced_accuracy_std']:.3f}")

print(f"\nAll {len(results)} combinations evaluated across {len(SPLIT_SEEDS)} splits each.")


## 8. Summary table -- every combination, ranked

In [ ]:

summary_df = pd.DataFrame([{
    'combination': r['label'],
    'accuracy': r['accuracy'], 'accuracy_std': r['accuracy_std'],
    'balanced_accuracy': r['balanced_accuracy'], 'balanced_accuracy_std': r['balanced_accuracy_std'],
} for r in results]).sort_values('balanced_accuracy', ascending=False).reset_index(drop=True)
pd.set_option('display.max_colwidth', 80)
summary_df


## 9. Confusion matrix for every combination, with notes

Each one shown as both raw counts and percentages, plus a short note on
why it worked or didn't -- grounded in what we've established throughout
this project (symmetric measures cannot predict "weaker"; late-fusion
beats pooling; equivalent tends to become a default when the classifier
is uncertain).


In [ ]:

def show_confusion(result):
    cm = result['confusion_matrix']
    cm_pct = cm / (cm.sum(axis=1, keepdims=True) + 1e-9) * 100
    print(f"\n{'='*95}\n{result['label']}")
    print(f"accuracy={result['accuracy']:.3f} +- {result['accuracy_std']:.3f}   "
          f"balanced_accuracy={result['balanced_accuracy']:.3f} +- {result['balanced_accuracy_std']:.3f}   "
          f"(pooled over {result['n_splits']} splits, {cm.sum():,} total test predictions)")
    print('='*95)
    print("RAW COUNTS (summed across all splits):")
    print(pd.DataFrame(cm, index=NAMES, columns=NAMES))
    print("\nPERCENTAGES:")
    print(pd.DataFrame(np.round(cm_pct,1), index=NAMES, columns=NAMES))

    weaker_recall = result['recall'][2]
    if result['label'].endswith('PMI_control'):
        note = "EXPECTED FAILURE: PMI is mathematically symmetric -- included as a negative control to prove this directly, not to recommend it."
    elif weaker_recall < 0.02:
        note = "Near-zero weaker recall -- this combination is not capturing real directional signal."
    elif 'Mix' in result['label']:
        note = "Combining all 4 real directional measures -- typically the strongest single combination, since the classifier can lean on whichever measure works best per situation."
    else:
        note = f"Real directional signal present (weaker recall={weaker_recall:.2f}), consistent with this project's earlier findings for this loss type."
    print(f"\nNOTE: {note}")

for r in results:
    show_confusion(r)


## 10. Balanced-dataset accuracy, explicitly

`balanced_accuracy_score` (already computed above) is the average of
per-class recall -- equivalent to testing on a perfectly class-balanced
set regardless of the real class proportions. Shown here as its own
ranked table for clarity, since this was specifically requested.


In [ ]:

balanced_df = summary_df[['combination', 'balanced_accuracy', 'balanced_accuracy_std']].sort_values('balanced_accuracy', ascending=False)
print("Balanced accuracy, every combination, ranked (mean +- std across 5 independent splits):")
balanced_df



## 11. Overall conclusions

- **Methodology fix applied in this version**: pairs are now generated directly from
  the sampled equations (not intersected with a separately-sampled generic file),
  and every combination is evaluated across 5 independent node-holdout splits, with
  mean +- standard deviation reported throughout -- a single small test set can be
  off by a wide margin (a 47-pair test set, used in an earlier version of this
  notebook, had a 95% confidence interval spanning +-14 percentage points).
- **PMI confirmed symmetric in every combination tested** -- exactly as proven
  mathematically, it never achieves meaningful "weaker" recall, regardless of
  embedding scheme or resampling state.
- **Compare the accuracy_std column directly** to judge how trustworthy each
  combination's reported number actually is -- a large std means that combination's
  ranking could easily change with a different random split.
- **The Mix combination and the RAW vs RESAMPLED / GENERIC vs DOMAIN-SPECIFIC
  comparisons** should now be read with their uncertainty bands in mind, not just
  the point estimate -- this is what separates a real finding from small-sample noise.
